# TSFM comparison — iso-budget heatmap (exploratory draft, not a numbered paper figure)

**Purpose**: analog of `main_fig3_heatmap.ipynb`, extended across the three
TSFM baselines (OSF, PhysioOmni, Mantis) plus SleepFM. See
`docs/npj_paper_md_files/TSFM_BASELINE_RESULTS_DRAFT.md` (Sections 6-7) for
the full writeup this notebook supports.

**Data**: same sources as `tsfm_fig_kvsk.ipynb` -- see that notebook's intro
cell for the loader details (real `load_heatmap` for SleepFM, the new
`load_heatmap_from_collected` for OSF/PhysioOmni/Mantis).

**Tasks**: sex, apnea, sleep efficiency, BMI -- the same four representative
tasks `main_fig3_heatmap.ipynb` itself uses.

**Head**: Transformer.

**Layout**: 5 rows (encoders) x 4 cols (tasks). **PhysioOmni x apnea will
render as "no data"** -- PhysioOmni has no respiratory-signal pathway
(Methods draft, Section 2 of the draft doc), so that cell is expected to be
empty, not a bug. This falls out automatically from `heatmap_panel`'s own
existing empty-DataFrame handling; nothing was special-cased for it.

**Not run end-to-end in this session** -- see the same note in
`tsfm_fig_kvsk.ipynb`.

In [ ]:
%matplotlib inline

In [ ]:
import sys
from pathlib import Path

# ── Workspace root: auto-detect by looking for final_results/ ─────────────────
def _find_workspace():
    """Walk up from CWD until we find a directory containing final_results/."""
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    # Explicit fallback (edit this if auto-detect fails)
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
FINAL_RESULTS  = WORKSPACE_ROOT / "final_results"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
# Exploratory draft output -- deliberately NOT final_npj/, so these never get
# mistaken for or accidentally overwrite an actual paper figure.
FINAL_OUT      = PAPER_FIGURES / "draft_tsfm"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# TSFM baselines' own collected/ roots (see docs/npj_paper_md_files/
# TSFM_BASELINE_RESULTS_DRAFT.md Section 7 for why these aren't under
# final_results/ like the paper's own SleepFM figures).
OSF_PHYSIOOMNI_COLLECTED = NSRR_TOOLS / "results" / "collected"
# Mantis results live in the SEPARATE NSRR-tools-mantis worktree, not this
# repo -- update this path if that worktree moves or is renamed, or once
# Mantis results are merged into this repo's own results/collected/.
MANTIS_COLLECTED = WORKSPACE_ROOT / "NSRR-tools-mantis" / "results" / "collected"

# Add utils to path (notebooks_npj/utils/)
_nb_dir = PAPER_FIGURES / "notebooks_npj"
sys.path.insert(0, str(_nb_dir))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS, BINARY_MAIN,
    HEAD_STYLE, TASK_LABEL, FONT_ANNOT, FONT_BASE, FONT_LABEL, FONT_TITLE,
)
from utils.data import set_root, load_analysis, load_heatmap, load_heatmap_from_collected
from utils import panels

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()

# ── Confirm the workspace root and both baseline result trees are found ───────
_ok_ws     = FINAL_RESULTS.exists()
_ok_osf    = (OSF_PHYSIOOMNI_COLLECTED / "phase0_osf" / "analysis.csv").exists()
_ok_pomni  = (OSF_PHYSIOOMNI_COLLECTED / "phase0_physioomni" / "analysis.csv").exists()
_ok_mantis = (MANTIS_COLLECTED / "phase0_mantis" / "analysis.csv").exists()
print(f"WORKSPACE_ROOT      : {WORKSPACE_ROOT}")
print(f"final_results/      : {'found' if _ok_ws else 'NOT FOUND -- edit _find_workspace() fallback'}")
print(f"phase0_osf          : {'found' if _ok_osf else 'NOT FOUND'}")
print(f"phase0_physioomni   : {'found' if _ok_pomni else 'NOT FOUND'}")
print(f"phase0_mantis       : {'found' if _ok_mantis else 'NOT FOUND -- is NSRR-tools-mantis cloned next to NSRR-tools?'}")

In [ ]:
# Panel labeling helper
def add_panel_label(ax, label, x=0.02, y=0.97):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="top", fontfamily="serif")

In [ ]:
# ── Encoders compared, in row order ──────────────────────────────────────────
# (row label, experiment name, collected_root or None)
#   collected_root=None   -> real paper figures' own loader (load_heatmap),
#                             reads final_results/{experiment}/inference/...
#   collected_root=<path> -> load_heatmap_from_collected (this session's new,
#                             purely-additive loader), reads
#                             {collected_root}/{experiment}/analysis.csv
#
# SleepFM appears TWICE (reduced- and full-channel) because OSF was run
# against the full-channel baseline and PhysioOmni/Mantis against the
# reduced-channel one (Methods draft, docs/npj_paper_md_files/
# TSFM_BASELINE_RESULTS_DRAFT.md Section 2) -- showing both avoids
# comparing OSF to the wrong SleepFM variant.
ENCODERS = [
    ("SleepFM (reduced-ch.)",    "phase0_v3",         None),
    ("SleepFM (full-ch.)",       "phase0_v3_full",    None),
    ("OSF (full-ch.)",           "phase0_osf",        OSF_PHYSIOOMNI_COLLECTED),
    ("PhysioOmni (reduced-ch.)", "phase0_physioomni", OSF_PHYSIOOMNI_COLLECTED),
    ("Mantis (reduced-ch.)",     "phase0_mantis",     MANTIS_COLLECTED),
]

def get_hmap(experiment, collected_root, task, head):
    if collected_root is None:
        return load_heatmap(experiment, task, head)
    return load_heatmap_from_collected(collected_root, experiment, task, head)

In [ ]:
HEAD   = "transformer"
TASKS  = ["sex_binary", "apnea_binary", "sleep_efficiency_binary", "bmi_binary"]
METRIC = "auroc"

hmaps = {
    (enc_label, task): get_hmap(experiment, collected_root, task, HEAD)
    for enc_label, experiment, collected_root in ENCODERS
    for task in TASKS
}
print({k: len(v) for k, v in hmaps.items()})

In [ ]:
N_ROWS, N_COLS = len(ENCODERS), len(TASKS)
ROW_H = 1.7  # inches per row

fig, axes = plt.subplots(N_ROWS, N_COLS,
                         figsize=(FULL_W + 3.0, N_ROWS * ROW_H + 0.4),
                         squeeze=False)

for r, (enc_label, experiment, collected_root) in enumerate(ENCODERS):
    for c, task in enumerate(TASKS):
        ax = axes[r][c]
        show_y = (c == 0)
        show_cbar_lbl = (c == N_COLS - 1)
        panels.heatmap_panel(ax, hmaps[(enc_label, task)], col=METRIC,
                             show_ylabels=show_y, show_cbar_label=show_cbar_lbl)
        if r == 0:
            ax.set_title(TASK_LABEL[task], fontsize=FONT_TITLE, pad=3)
        else:
            ax.set_title("")
        if show_y:
            ax.set_ylabel(f"{enc_label}\n" + ax.get_ylabel(), fontsize=FONT_LABEL)
        add_panel_label(ax, f"({chr(97 + r * N_COLS + c)})")

fig.tight_layout(h_pad=0.6, w_pad=0.5)
plt.show()

In [ ]:
# ── Run this cell once the figure looks good ──────────────────────────────
save_figure(fig, FINAL_OUT, "tsfm_fig_heatmap")
print("Saved (exploratory, not copied to npj_digital_medicine_submission/) ->",
      FINAL_OUT / "tsfm_fig_heatmap.pdf")
print("If you decide to use this, rename/restyle it into main or")
print("supplementary format yourself, same as the paper's own convention")
print("for main_fig*/sfig* filenames.")